# MetroPT temporal representation experiment

This notebook executes the predeclared XGBoost → TCN → Attention-TCN comparison. It first verifies the real model environment without examining July outcomes, then runs all 36 model/horizon/seed cells. Completed cells are hash-checked and resumed from Google Drive. Do not change the frozen configuration after seeing results.

## 1. Runtime and repository
Use a **T4 GPU** runtime. The repository is synchronized to the experiment branch and the exact revision is recorded in every cell manifest.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

repo = Path('/content/metropt3-predictive-maintenance')
branch = 'investigation/temporal-validation'
if repo.exists():
    subprocess.run(['git', 'fetch', 'origin', branch], cwd=repo, check=True)
    subprocess.run(['git', 'checkout', branch], cwd=repo, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{branch}'], cwd=repo, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', branch, '--single-branch', 'https://github.com/SahilBh01r1769/metropt3-predictive-maintenance.git', str(repo)], check=True)
os.chdir(repo)
source_root = str(repo / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
os.environ['METROPT_CODE_REVISION'] = revision
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-experiment.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
import importlib.util
assert importlib.util.find_spec('metropt3') is not None, 'Repository package is not importable after setup'
print('Experiment revision:', revision)
print('Package source:', importlib.util.find_spec('metropt3').origin)

## 2. Persistent checkpoints
Authorize Google Drive when prompted. Model artifacts remain there so a disconnected session can restart and skip every hash-verified completed cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
output_root = Path('/content/drive/MyDrive/metropt_temporal_run_v1')
output_root.mkdir(parents=True, exist_ok=True)
print('Persistent run directory:', output_root)

## 3. Dependency and architecture pilot
This pilot uses synthetic tensors only. It checks XGBoost availability, output shapes, normalized attention weights, and the causal encoder property before any held-out probabilities are generated.

In [ ]:
import numpy as np
import torch
from xgboost import XGBClassifier
from metropt3.experiment_config import load_experiment_config
from metropt3.temporal_models import build_temporal_model

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then run again.'
config = load_experiment_config()
sample = torch.randn(3, config['sequence']['timesteps'], 9)
changed_future = sample.clone(); changed_future[:, 60:, :] += 50
for name in ('tcn', 'attention_tcn'):
    model = build_temporal_model(name, input_channels=9, config=config).eval()
    with torch.no_grad():
        encoded = model.encoder(sample.transpose(1, 2))
        changed = model.encoder(changed_future.transpose(1, 2))
        assert torch.allclose(encoded[:, :, :60], changed[:, :, :60], atol=1e-6), 'Encoder is not causal'
        logits, weights = model(sample, return_attention=True)
        assert logits.shape == (3,)
        if name == 'attention_tcn':
            assert weights.shape == (3, 120)
            assert torch.allclose(weights.sum(1), torch.ones(3), atol=1e-6)
xgb = XGBClassifier(n_estimators=2, tree_method='hist', random_state=42)
xgb.fit(np.arange(40).reshape(20, 2), np.array([0, 1] * 10))
assert xgb.predict_proba([[1, 2]]).shape == (1, 2)
print('Pilot passed on', torch.cuda.get_device_name(0))

## 4. Build the audited experiment dataset once
The downloaded file must match the frozen audit hash. Validation, continuity segmentation, feature windows, active-failure quarantine and 30-second sequence construction all reuse repository code.

In [ ]:
import pandas as pd
from metropt3.audit import file_sha256
from metropt3.config import RAW_FILENAME
from metropt3.features import build_windows
from metropt3.labels import add_failure_labels
from metropt3.sequences import build_sequence_dataset
from metropt3.validation import validate_and_segment

subprocess.run([sys.executable, 'scripts/download_data.py'], check=True)
csv_path = Path('data') / RAW_FILENAME
actual_hash = file_sha256(csv_path)
assert actual_hash == config['audit_evidence']['dataset_sha256'], 'Dataset differs from audited source'
raw = pd.read_csv(csv_path)
valid, validation = validate_and_segment(raw)
windows = build_windows(valid)
quarantine_labels = add_failure_labels(windows, horizon_hours=12)
dataset = build_sequence_dataset(valid, quarantine_labels, config=config)
assert len(dataset.metadata) == 7846, f'Expected 7,846 predictive windows, found {len(dataset.metadata):,}'
context = {
    'code_revision': revision,
    'dataset_sha256': actual_hash,
    'validated_rows': len(valid),
    'quarantined_rows': validation.quarantined_rows,
    'feature_windows': len(windows),
    'predictive_sequence_windows': len(dataset.metadata),
    'sequence_shape': list(dataset.values.shape),
    'mean_bin_observed_fraction': float(dataset.values[:, :, -1].mean()),
    'device': torch.cuda.get_device_name(0),
}
(output_root / 'run_context.json').write_text(json.dumps(context, indent=2))
context

## 5. Run or resume all 36 predeclared cells
Do not edit the configuration based on intermediate scores. If Colab disconnects, reconnect and run all cells again; verified cells are skipped.

In [ ]:
from metropt3.experiment_runner import run_experiment
run_counts = run_experiment(dataset, output_root, config=config)
print('This session:', run_counts)

## 6. Validate and export evidence
Validation fails rather than packaging partial, mismatched, non-finite or missing evidence. Model binaries remain in Drive; the downloaded ZIP contains the smaller auditable traces, histories, metrics, manifests and attention weights needed for analysis.

In [ ]:
from metropt3.experiment_evidence import validate_and_summarize_experiment
metrics, report = validate_and_summarize_experiment(
    output_root,
    output_csv=output_root / 'metrics.csv',
    report_path=output_root / 'validation.json',
    config=config,
)
assert report['valid'] and report['complete_cells'] == 36
bundle = Path('/content/metropt_temporal_evidence.zip')
allowed_suffixes = {'.csv', '.json', '.npy'}
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in output_root.rglob('*'):
        if path.is_file() and path.suffix in allowed_suffixes:
            archive.write(path, arcname=path.relative_to(output_root))
print('Validated cells:', report['complete_cells'])
print('Metric rows:', len(metrics))
print('Evidence bundle:', bundle, f'({bundle.stat().st_size / 1_000_000:.1f} MB)')
from google.colab import files
files.download(str(bundle))